# 5. Modified Policy Iteration
### Combines Partial Evaluation and Policy Improvement

**Modified Policy Iteration (MPI)** bridges the gap between **Policy Iteration** (which evaluates to full convergence) and **Value Iteration** (which truncates after $k=1$ evaluation step).

#### Key Concepts:
1. **Generalized Policy Iteration (GPI) Spectrum**:
   - $k = \infty$: Standard Policy Iteration.
   - $k = 1$: Standard Value Iteration.
   - $1 < k < \infty$: Modified Policy Iteration.
2. **Computational Benefit**:
   Executing $k$ (e.g. 3 to 10) successive evaluation updates without full convergence often converges much faster in wall-clock time than full Policy Iteration while avoiding the slow propagation of single-sweep Value Iteration.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Step 2: Define a 4x4 GridWorld MDP Environment
# States: 16 grid cells (0 to 15), where (0,0) [state 0] and (3,3) [state 15] are terminal
# Actions: 0: UP, 1: RIGHT, 2: DOWN, 3: LEFT

class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.num_states = size * size
        self.num_actions = 4
        self.actions = ['UP', 'RIGHT', 'DOWN', 'LEFT']
        self.terminal_states = {0, self.num_states - 1}
        
    def get_next_state_and_reward(self, state, action):
        if state in self.terminal_states:
            return state, 0.0
            
        r, c = divmod(state, self.size)
        if action == 0:    # UP
            r = max(r - 1, 0)
        elif action == 1:  # RIGHT
            c = min(c + 1, self.size - 1)
        elif action == 2:  # DOWN
            r = min(r + 1, self.size - 1)
        elif action == 3:  # LEFT
            c = max(c - 1, 0)
            
        next_state = r * self.size + c
        reward = -1.0  # -1 step cost until terminal goal reached
        return next_state, reward

env = GridWorld()
print(f"GridWorld initialized with {env.num_states} states and {env.num_actions} actions.")


In [ ]:
# Step 3: Implement Modified Policy Iteration
def modified_policy_iteration(env, m_eval_steps=3, gamma=1.0, theta=1e-4):
    V = np.zeros(env.num_states)
    policy = np.ones((env.num_states, env.num_actions)) / env.num_actions
    
    total_eval_sweeps = 0
    outer_iterations = 0
    
    while True:
        outer_iterations += 1
        
        # 1. Truncated Policy Evaluation (run for fixed m_eval_steps)
        for _ in range(m_eval_steps):
            V_new = np.copy(V)
            for s in range(env.num_states):
                if s in env.terminal_states:
                    continue
                v_s = sum(
                    policy[s][a] * (env.get_next_state_and_reward(s, a)[1] + gamma * V[env.get_next_state_and_reward(s, a)[0]])
                    for a in range(env.num_actions)
                )
                V_new[s] = v_s
            V = V_new
            total_eval_sweeps += 1
            
        # 2. Greedy Policy Improvement and Convergence Check
        delta = 0.0
        policy_stable = True
        
        for s in range(env.num_states):
            if s in env.terminal_states:
                continue
                
            old_a = np.argmax(policy[s])
            action_values = [
                env.get_next_state_and_reward(s, a)[1] + gamma * V[env.get_next_state_and_reward(s, a)[0]]
                for a in range(env.num_actions)
            ]
            
            best_a = np.argmax(action_values)
            delta = max(delta, abs(V[s] - action_values[best_a]))
            
            if old_a != best_a:
                policy_stable = False
                
            policy[s] = np.zeros(env.num_actions)
            policy[s][best_a] = 1.0
            
        if delta < theta or policy_stable:
            print(f"MPI (m={m_eval_steps}) converged in {outer_iterations} outer rounds ({total_eval_sweeps} total sweeps).")
            break
            
    return policy, V

mpi_policy, mpi_V = modified_policy_iteration(env, m_eval_steps=4)


In [ ]:
# Step 4: Comparison across different truncation values (k = 1, 3, 10, full)
k_values = [1, 2, 4, 8, 16]
results = []

for k in k_values:
    # Measure steps
    V_test = np.zeros(env.num_states)
    pol_test = np.ones((env.num_states, env.num_actions)) / env.num_actions
    sweeps = 0
    rounds = 0
    while True:
        rounds += 1
        for _ in range(k):
            V_n = np.copy(V_test)
            for s in range(env.num_states):
                if s in env.terminal_states: continue
                V_n[s] = sum(pol_test[s][a] * (env.get_next_state_and_reward(s, a)[1] + 1.0 * V_test[env.get_next_state_and_reward(s, a)[0]]) for a in range(env.num_actions))
            V_test = V_n
            sweeps += 1
        stable = True
        for s in range(env.num_states):
            if s in env.terminal_states: continue
            old_a = np.argmax(pol_test[s])
            act_vals = [env.get_next_state_and_reward(s, a)[1] + 1.0 * V_test[env.get_next_state_and_reward(s, a)[0]] for a in range(env.num_actions)]
            best_a = np.argmax(act_vals)
            if old_a != best_a: stable = False
            pol_test[s] = np.zeros(env.num_actions)
            pol_test[s][best_a] = 1.0
        if stable:
            break
    results.append((k, rounds, sweeps))

print("K (Evaluation sweeps per round) | Outer Rounds | Total Sweeps")
print("-" * 55)
for k, r, s in results:
    print(f"{k:^31} | {r:^12} | {s:^12}")


In [ ]:
# Function to plot State-Value Heatmap and Policy Arrows
def plot_grid(V, policy=None, title="State-Value Grid"):
    fig, ax = plt.subplots(figsize=(6, 6))
    grid_v = V.reshape((4, 4))
    
    cax = ax.matshow(grid_v, cmap='Blues_r')
    fig.colorbar(cax)
    
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            val_text = f"{V[s]:.1f}"
            if s in [0, 15]:
                val_text += "\n(Goal)"
            elif policy is not None:
                best_a = np.argmax(policy[s])
                val_text += f"\n{arrows[best_a]}"
                
            ax.text(c, r, val_text, ha='center', va='center', color='black', fontsize=11, weight='bold')
            
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    plt.title(title, pad=20)
    plt.show()


In [ ]:
# Step 6: Visualize Resulting Optimal Policy and State Values
plot_grid(mpi_V, policy=mpi_policy, title="Modified Policy Iteration: Optimal Policy & Values")


### Summary
- Generalizes Policy Iteration and Value Iteration into a unified family of algorithms.
- Balances the computational cost of evaluation sweeps against the benefit of greedy policy updates.
